# 06 - Data Assessment

## Purpose
Load all processed datasets and describe them. Shape, dtypes, nulls, value
ranges, country name formats, CN code formats, and cross-dataset consistency.
No cleaning or transformation. Observation and notes only.

## Inputs
- `data/processed/cbam_defaults.csv`
- `data/processed/steel_route_intensity.csv`
- `data/processed/hydrogen_route_intensities.csv`
- `data/processed/eu_import_trade_flows.csv`
- `data/processed/country_grid_electricity.csv`

## Notes
- Findings from this notebook feed directly into `07_clean_and_align.ipynb`.
- Each section covers one dataset, followed by a cross-dataset section
  focused on join readiness.

In [45]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

processed = Path("../data/processed")

defaults  = pd.read_csv(processed / "cbam_defaults.csv")
flows     = pd.read_csv(processed / "eu_import_trade_flows.csv")
grid      = pd.read_csv(processed / "country_grid_electricity.csv")
hydrogen  = pd.read_csv(processed / "hydrogen_route_intensities.csv")
steel     = pd.read_csv(processed / "steel_route_intensity.csv")

datasets = {
    "cbam_defaults":              defaults,
    "eu_import_trade_flows":      flows,
    "country_grid_electricity":   grid,
    "hydrogen_route_intensities": hydrogen,
    "steel_route_intensity":      steel,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

cbam_defaults: (10671, 10)
eu_import_trade_flows: (165182, 9)
country_grid_electricity: (193936, 10)
hydrogen_route_intensities: (6, 5)
steel_route_intensity: (12, 4)


## 1. CBAM Defaults 

(`cbam_defaults.csv`)

119 countries, all CBAM-covered CN codes. One row per country/CN code/description
combination. Emission values in tCO2 per tonne of product.

In [3]:
# Basic structure check
print("Shape:", defaults.shape)
print("\nDtypes:")
print(defaults.dtypes)
print("\nSample:")
defaults.head(3)

Shape: (10671, 10)

Dtypes:
country                     str
cn_code                     str
description                 str
direct_emissions        float64
indirect_emissions      float64
total_emissions         float64
default_2026            float64
default_2027            float64
default_2028_onwards    float64
production_route            str
dtype: object

Sample:


,country,cn_code,description,direct_emissions,indirect_emissions,total_emissions,default_2026,default_2027,default_2028_onwards,production_route
0,Albania,2523 10 00,Grey clinker,0.8700,0.0000,0.8700,0.9570,1.0440,1.1310,(A)
1,Albania,2523 29 00,Grey Portland cement,0.9000,0.0300,0.9300,1.0230,1.1160,1.2090,NaN
2,Albania,2523 90 00,Grey hydraulic cements,0.8600,0.0300,0.8900,0.9790,1.0680,1.1570,(A)


In [4]:
# Two columns are expected to have nulls:
# - indirect_emissions: electricity and hydrogen have no indirect component under CBAM methodology
# - production_route: only populated for iron/steel rows where the xlsx specifies a route
null_counts = defaults.isnull().sum()
null_pct = (null_counts / len(defaults) * 100).round(1)
print(pd.DataFrame({"null_count": null_counts, "null_pct": null_pct}).to_string())

                      null_count  null_pct
country                        0    0.0000
cn_code                        0    0.0000
description                    0    0.0000
direct_emissions               0    0.0000
indirect_emissions          7926   74.3000
total_emissions                0    0.0000
default_2026                   1    0.0000
default_2027                   0    0.0000
default_2028_onwards           0    0.0000
production_route            1033    9.7000


In [5]:
# Each country should have the same set of CN codes, so row counts should be roughly equal.
# Outliers may indicate missing rows for a country or genuine scope differences in the source xlsx.
countries = sorted(defaults["country"].unique())
print("Distinct countries:", len(countries))

rows_per_country = defaults["country"].value_counts()
print("\nRow count distribution across countries:")
print(rows_per_country.describe())

print("\nCountries with fewer rows than the median:")
print(rows_per_country[rows_per_country < rows_per_country.median()])

Distinct countries: 119

Row count distribution across countries:
count   119.0000
mean     89.6723
std      97.7892
min       1.0000
25%      28.0000
50%      51.0000
75%     220.5000
max     259.0000
Name: count, dtype: float64

Countries with fewer rows than the median (possible missing CN codes):
country
United Arab Emirates               34
Dominican Republic                 33
Georgia                            32
Lebanon                            32
Belarus                            31
Cuba                               31
Iraq                               31
Libya                              31
North Korea                        31
Trinidad and Tobago                31
Turkmenistan                       31
Albania                            30
Kyrgyzstan                         30
Madagascar                         30
Mali                               30
Oman                               30
Singapore                          30
Sudan                              30
Yemen 

In [6]:
# CN codes are stored as spaced strings (e.g. "2523 10 00").
# Codes appear at 4, 6, and 8-digit lengths, all valid under the CBAM regulation.
# Stripping spaces is needed before any join with trade flow data (which stores CN codes as integers).
defaults["cn_stripped"] = defaults["cn_code"].str.replace(" ", "")
defaults["cn_len"] = defaults["cn_stripped"].str.len()

print("CN code length distribution (after stripping spaces):")
print(defaults["cn_len"].value_counts().sort_index())

print("\nExample of each length:")
for length in sorted(defaults["cn_len"].unique()):
    sample = defaults[defaults["cn_len"] == length]["cn_code"].iloc[0]
    print(f"  {length}-digit: {sample!r}")

CN code length distribution (after stripping spaces):
cn_len
4    1111
6    1960
8    7600
Name: count, dtype: int64

Example of each length:
  4-digit: '7601'
  6-digit: '761090'
  8-digit: '2523 10 00'


In [7]:
# Descriptive stats across all emission columns.
# Key check: total_emissions should equal direct + indirect for all rows where indirect is not null.
# Any mismatch suggests a parsing error in extraction.
emission_cols = ["direct_emissions", "indirect_emissions", "total_emissions",
                 "default_2026", "default_2027", "default_2028_onwards"]
print("Emission value ranges (tCO2/t):")
print(defaults[emission_cols].describe().to_string())

print("\nZero and negative values (unexpected in emission defaults):")
for col in ["direct_emissions", "total_emissions"]:
    zeros = (defaults[col] == 0).sum()
    negs  = (defaults[col] < 0).sum()
    print(f"  {col}: {zeros} zeros, {negs} negatives")

check = defaults.dropna(subset=["indirect_emissions"]).copy()
check["diff"] = (check["total_emissions"] - (check["direct_emissions"] + check["indirect_emissions"])).abs()
mismatches = check[check["diff"] > 0.01]
print(f"\nRows where total != direct + indirect (tolerance 0.01): {len(mismatches)}")
if len(mismatches) > 0:
    print(mismatches[["country", "cn_code", "direct_emissions",
                       "indirect_emissions", "total_emissions", "diff"]].to_string())

Emission value ranges (tCO2/t):
       direct_emissions  indirect_emissions  total_emissions  default_2026  default_2027  default_2028_onwards
count        10671.0000           2745.0000       10671.0000    10670.0000    10671.0000            10671.0000
mean             2.5650              0.0743           2.5840        2.8150        3.0429                3.2709
std              1.8128              0.0382           1.8004        1.9938        2.1901                2.3880
min              0.0000              0.0000           0.0000        0.0000        0.0000                0.0000
25%              1.3900              0.0500           1.4400        1.5290        1.5960                1.6770
50%              2.3130              0.0700           2.3300        2.5410        2.7720                3.0030
75%              3.2100              0.1000           3.2100        3.5310        3.8520                4.1730
max             26.6400              0.3500          26.6400       29.3040      

In [8]:
# The CBAM regulation applies a fixed markup to total_emissions to derive default values:
# 10% for 2026, 20% for 2027, 30% for 2028 onwards, for most sectors.
# Fertilizers use a 1% markup. Checking the actual multipliers confirms extraction was clean
# and identifies any sectors with different markup schedules.
sample = defaults.dropna(subset=["default_2026"]).copy()
sample["mult_2026"] = (sample["default_2026"] / sample["total_emissions"]).round(4)
sample["mult_2027"] = (sample["default_2027"] / sample["total_emissions"]).round(4)
sample["mult_2028"] = (sample["default_2028_onwards"] / sample["total_emissions"]).round(4)

print("Uplift multipliers by sector (distinct combinations):")
print(sample.groupby(["mult_2026", "mult_2027", "mult_2028"]).size()
      .reset_index(name="row_count").sort_values("row_count", ascending=False).to_string(index=False))

Uplift multipliers by sector (distinct combinations):
 mult_2026  mult_2027  mult_2028  row_count
    1.1000     1.2000     1.3000       8275
    1.0100     1.0100     1.0100       2389
    1.1000     1.2100     1.3310          5


In [9]:
# production_route is only populated for iron/steel rows.
# Values correspond to production route codes defined in the CBAM regulation annex.
# Null = sector does not use route-based benchmarking (cement, fertilizers, hydrogen, electricity).
print("Production route values (null = non-steel sector):")
print(defaults["production_route"].value_counts(dropna=False).to_string())

Production route values (null = non-steel sector):
production_route
           3452
(C)        2675
NaN        1033
(F)         884
(L)         864
(K)         720
(E)         515
(A)         171
(H)         170
(C)/(F)     130
(B)          32
(E)/(H)      25


### 1. Observations

**Nulls**
- `indirect_emissions` is null for 74.3% of rows (7,926). Expected: electricity and
  hydrogen CN codes carry no indirect component under CBAM methodology.
- `production_route` is null for 9.7% of rows (1,033). Expected: only iron/steel rows
  carry a route code. Non-null values are letter codes (A, B, C, etc.) referencing
  production route definitions in the CBAM regulation annex.
- `default_2026` has 1 null. Needs investigation in cleaning — likely a row where the
  source xlsx held a dash or placeholder rather than a numeric value.

**Country coverage**
- 119 countries. Row counts vary significantly (min 1, max 259, median 51). The high
  variance is expected: larger industrial exporters have more CN codes covered. However,
  countries with very low row counts (Angola 9, Congo 4, Jamaica 4, several at 1-3)
  should be verified — they may reflect genuine scope limits in the source xlsx or
  extraction gaps.
- `Democratic Republic of the Cong` is a truncated country name, almost certainly
  "Democratic Republic of the Congo". Flag for standardization in cleaning.
- `Myanmar_Burma` uses an underscore separator rather than a slash or "and". Standardize
  in cleaning.
- `Côte d'Ivoire` and `Curaçao` contain non-ASCII characters. Confirm these survive
  encoding round-trips correctly.

**CN code format**
- Stored as spaced strings (e.g. `"2523 10 00"`). Three lengths present after stripping
  spaces: 4-digit (1,111 rows), 6-digit (1,960 rows), 8-digit (7,600 rows). All valid
  under the CN nomenclature. Spaces must be stripped before any join with trade flow data,
  which stores CN codes as integers.

**Emission values**
- 533 rows where `total_emissions != direct + indirect` at a 0.01 tolerance. All
  differences are exactly 0.01, spread across many countries and sectors. This is a
  rounding artifact from the source xlsx, not an extraction error — the values were
  published at 2 decimal places and the sum of two 2dp values occasionally rounds
  differently than the stored total. Not a data quality issue, but worth documenting.
  No action needed in cleaning.
- 1 row where `direct_emissions` and `total_emissions` are both 0. Investigate in
  cleaning.

**Uplift multipliers**
- Three distinct markup schedules confirmed:
  - 10/20/30% (×1.10/1.20/1.30): 8,275 rows — standard sectors (cement, steel, aluminium)
  - 1/1/1% (×1.01/1.01/1.01): 2,389 rows — fertilizers
  - 10/21/33.1% (×1.10/1.21/1.331): 5 rows — compounded markup, investigate which
    CN codes these are
- Extraction is clean. Multipliers are consistent with the regulation.

## 2. Steel Route Intensities 

(`steel_route_intensity.csv`)

Global average CO2 and energy intensities by production route, manually
transcribed from the Worldsteel Sustainability Indicators Report 2025.
Reference table only — no country dimension. Used to benchmark CBAM default
emission values against global production route averages.

In [14]:
# Small reference table — print in full rather than sampling
print("Shape:", steel.shape)
print("\nDtypes:")
print(steel.dtypes)
print("\nFull table:")
print(steel.to_string(index=False))

Shape: (12, 4)

Dtypes:
production_route                 str
year                           int64
co2_intensity_tco2_per_t     float64
energy_intensity_gj_per_t    float64
dtype: object

Full table:
production_route  year  co2_intensity_tco2_per_t  energy_intensity_gj_per_t
          BF-BOF  2022                    2.3300                    23.9800
          BF-BOF  2023                    2.3300                    24.2400
          BF-BOF  2024                    2.3400                    23.8800
       Scrap-EAF  2022                    0.6700                    10.1300
       Scrap-EAF  2023                    0.6900                    10.2100
       Scrap-EAF  2024                    0.6900                     9.8400
         DRI-EAF  2022                    1.3600                    22.2500
         DRI-EAF  2023                    1.4300                    23.1300
         DRI-EAF  2024                    1.4700                    23.3000
      Global avg  2022                   

In [ ]:
# Examine nulls and value ranges
print("Nulls:")
print(steel.isnull().sum())

print("\nCO2 intensity (tCO2/t) by route:")
print(steel.groupby("production_route")["co2_intensity_tco2_per_t"]
      .agg(["min", "max", "mean"]).round(3))

print("\nYear-on-year stability (std dev across 2022-2024):")
print(steel.groupby("production_route")["co2_intensity_tco2_per_t"]
      .std().round(4))

Nulls:
production_route             0
year                         0
co2_intensity_tco2_per_t     0
energy_intensity_gj_per_t    0
dtype: int64

CO2 intensity (tCO2/t) by route:
                    min    max   mean
production_route                     
BF-BOF           2.3300 2.3400 2.3330
DRI-EAF          1.3600 1.4700 1.4200
Global avg       1.9200 1.9200 1.9200
Scrap-EAF        0.6700 0.6900 0.6830

Year-on-year stability (std dev across 2022-2024):
production_route
BF-BOF       0.0058
DRI-EAF      0.0557
Global avg   0.0000
Scrap-EAF    0.0115
Name: co2_intensity_tco2_per_t, dtype: float64


In [16]:
# The production_route codes in cbam_defaults map to three steelmaking routes:
# (C),(F) = BF/BOF primary routes -> maps to "BF-BOF" in Worldsteel
# (D),(G) = DRI/EAF routes        -> maps to "DRI-EAF" in Worldsteel
# (E),(H),(J) = Scrap/EAF routes  -> maps to "Scrap-EAF" in Worldsteel
# "Global avg" is a Worldsteel summary row, not a joinable route.

worldsteel_routes = set(steel["production_route"].unique())
cbam_route_codes = set(defaults["production_route"].dropna().unique())

print("Worldsteel routes:", worldsteel_routes)
print("\nCBAM default route codes:", cbam_route_codes)
print("\nExpected mapping:")
mapping = {
    "BF-BOF":    ["(C)", "(F)", "(C)/(F)"],
    "DRI-EAF":   ["(D)", "(G)"],
    "Scrap-EAF": ["(E)", "(H)", "(J)", "(E)/(H)"],
}
for route, codes in mapping.items():
    count = defaults[defaults["production_route"].isin(codes)].shape[0]
    print(f"  {route} <- {codes}: {count} rows in cbam_defaults")

Worldsteel routes: {'BF-BOF', 'Scrap-EAF', 'Global avg', 'DRI-EAF'}

CBAM default route codes: {'(E)/(H)', '(L)', '\xa0', '(K)', '(E)', '(H)', '(B)', '(F)', '(A)', '(C)/(F)', '(C)'}

Expected mapping:
  BF-BOF <- ['(C)', '(F)', '(C)/(F)']: 3689 rows in cbam_defaults
  DRI-EAF <- ['(D)', '(G)']: 0 rows in cbam_defaults
  Scrap-EAF <- ['(E)', '(H)', '(J)', '(E)/(H)']: 710 rows in cbam_defaults


### 2. Observations

**Structure**
- 12 rows, no nulls. Three production routes across 2022-2024 plus a
  Global avg summary row. Clean, manually transcribed reference table.

**Value stability**
- BF-BOF and Scrap-EAF are essentially flat across 2022-2024 (std dev
  0.006 and 0.012 respectively), confirming these are reliable reference
  values rather than volatile annual measurements.
- DRI-EAF shows a more notable upward trend (1.36 to 1.47 tCO2/t, std dev
  0.056). Worth noting when choosing which year to use as the join value
  in the analysis layer.
- Global avg std dev is exactly 0.000 — the 1.92 figure is constant across
  all three years, suggesting it is a rounded or fixed reference figure
  rather than a recalculated annual average.

**Route name alignment with CBAM defaults**
- BF-BOF maps to codes (C), (F), (C)/(F): 3,689 rows in cbam_defaults.
- Scrap-EAF maps to codes (E), (H), (J), (E)/(H): 710 rows in cbam_defaults.
- DRI-EAF maps to codes (D) and (G): 0 rows found. Neither code appears
  in the defaults data at all. This means no country in the CBAM defaults
  has been assigned a DRI-EAF production route, consistent with the
  industry reporting that the vast majority of origins were classified as
  BF-BOF regardless of actual route.
- Two unexpected values in cbam_defaults production_route: `(L)` and `(K)`
  (aluminium codes) appearing alongside steel codes confirms the column
  covers all CBAM sectors, not steel only. `'\xa0'` is a non-breaking space
  character — a data quality issue to fix in 07.
- Global avg is not a joinable route. Exclude from any join with
  cbam_defaults.

## 3. Hydrogen Route Intensities 

(`hydrogen_route_intensities.csv`)

Global average GHG emission intensities per hydrogen production route,
extracted from JRC135067. Single CN code (2804 10 00). No country dimension.

In [17]:
# Small reference table — print in full
print("Shape:", hydrogen.shape)
print("\nDtypes:")
print(hydrogen.dtypes)
print("\nFull table:")
print(hydrogen.to_string(index=False))

Shape: (6, 5)

Dtypes:
cn_code                             str
feedstock_type                      str
total_emissions_tco2_per_th2    float64
comments                            str
source                              str
dtype: object

Full table:
   cn_code                      feedstock_type  total_emissions_tco2_per_th2                                                                                                           comments               source
2804 10 00                         Natural gas                        9.0000                                                                                                                NaN             IEA 2023
2804 10 00                                Coal                       19.2000                                                                                                                NaN             IEA 2023
2804 10 00                Naphtha (by-product)                        7.0000                        Value consi

In [18]:
# Examine nulls and value ranges
print("Nulls:")
print(hydrogen.isnull().sum())

print("\nEmission intensity range (tCO2/tH2):")
print(hydrogen["total_emissions_tco2_per_th2"].describe())

print("\nFeedstock types:")
print(hydrogen["feedstock_type"].value_counts())

Nulls:
cn_code                         0
feedstock_type                  0
total_emissions_tco2_per_th2    0
comments                        3
source                          0
dtype: int64

Emission intensity range (tCO2/tH2):
count    6.0000
mean    12.8833
std      6.7730
min      7.0000
25%      7.5000
50%     10.5000
75%     17.4000
max     23.1000
Name: total_emissions_tco2_per_th2, dtype: float64

Feedstock types:
feedstock_type
Natural gas                            1
Coal                                   1
Naphtha (by-product)                   1
Oil                                    1
Electrolysis (chlor-alkali)            1
Electrolysis (water, world average)    1
Name: count, dtype: int64


In [20]:
# Confirm hydrogen CN code is present in cbam_defaults
h2_cn = hydrogen["cn_code"].iloc[0].replace(" ", "")
match = defaults[defaults["cn_code"].str.replace(" ", "") == h2_cn]
print(f"Hydrogen CN code: {hydrogen['cn_code'].iloc[0]}")
print(f"Rows in cbam_defaults for this CN code: {len(match)}")
print("\nSample:")
print(match[["country", "cn_code", "description",
             "direct_emissions", "total_emissions",
             "production_route"]].head(5).to_string(index=False))

Hydrogen CN code: 2804 10 00
Rows in cbam_defaults for this CN code: 93

Sample:
  country    cn_code description  direct_emissions  total_emissions production_route
  Albania 2804 10 00    Hydrogen           14.0300          14.0300              NaN
  Algeria 2804 10 00    Hydrogen           10.8200          10.8200              NaN
   Angola 2804 10 00    Hydrogen           10.8200          10.8200              NaN
Argentina 2804 10 00    Hydrogen           10.8200          10.8200                 
Australia 2804 10 00    Hydrogen           10.8200          10.8200                 


### 3. Observations

**Structure**
- 6 rows, no nulls except `comments` (3 nulls, expected — most routes have
  no qualifying comment). Single CN code (2804 10 00), one row per feedstock
  type. Clean reference table.

**Value ranges**
- Emission intensities range from 7.0 (Naphtha, Electrolysis chlor-alkali)
  to 23.1 tCO2/tH2 (Electrolysis water, world average). The wide spread
  reflects fundamentally different production processes rather than data
  quality issues.
- Notably, water electrolysis at 23.1 tCO2/tH2 is higher than all fossil
  fuel routes due to the current global grid emission intensity. This
  reverses as grids decarbonize — analytically relevant for the dashboard.

**Alignment with cbam_defaults**
- CN code 2804 10 00 is present in cbam_defaults with 93 rows (one per
  country). All rows have null `production_route`, as expected — hydrogen
  does not use the letter code system.
- cbam_defaults stores a single country-level emission intensity per
  country for hydrogen, not broken down by feedstock type. The JRC135067
  figures are global averages by route and cannot be directly joined to
  the country-level defaults without additional methodology. No join
  attempted here.
- Argentina has a non-null but blank `production_route` value (empty
  string or whitespace) rather than null. Consistent with the `'\xa0'`
  finding in section 2 — flag for cleaning in 07.


## 4. EU Import Trade Flows 

(`eu_import_trade_flows.csv`)

COMEXT EU27 import flows by CN code, partner country and year (2020-2024),
pulled via the Eurostat API (dataset DS-045409). Value in euros and quantity
in tonnes where available. One row per (partner, product, indicator, year)
combination.

In [21]:
# Basic structure check — confirm expected columns and data types
print("Shape:", flows.shape)
print("\nDtypes:")
print(flows.dtypes)
print("\nSample:")
flows.head(3)

Shape: (165182, 9)

Dtypes:
freq             str
reporter         str
partner          str
product        int64
flow           int64
indicator        str
year           int64
value        float64
material         str
dtype: object

Sample:


,freq,reporter,partner,product,flow,indicator,year,value,material
0,A,EU27_2020,AR,26011200,1,VALUE_IN_EUROS,2020,30.0000,iron_steel
1,A,EU27_2020,BE,26011200,1,VALUE_IN_EUROS,2020,3333059.0000,iron_steel
2,A,EU27_2020,BR,26011200,1,VALUE_IN_EUROS,2020,45547476.0000,iron_steel


In [22]:
# Confirm structural fields contain only expected values.
# reporter should always be EU27_2020 (single reporter dataset).
# flow should be 1 (imports only) — confirm no export rows slipped in.
# indicator distinguishes value (EUR) from quantity (tonnes) rows.
print("Nulls:")
print(flows.isnull().sum())

print("\nreporter values:", flows["reporter"].unique())
print("flow values (1=import, 2=export):", flows["flow"].unique())
print("freq values:", flows["freq"].unique())

print("\nIndicator breakdown:")
print(flows["indicator"].value_counts())

print("\nMaterial breakdown:")
print(flows["material"].value_counts())

print("\nYears present:", sorted(flows["year"].unique()))
print("\nRows per year:")
print(flows["year"].value_counts().sort_index())

Nulls:
freq           0
reporter       0
partner      178
product        0
flow           0
indicator      0
year           0
value          0
material       0
dtype: int64

reporter values: <StringArray>
['EU27_2020']
Length: 1, dtype: str
flow values (1=import, 2=export): [1]
freq values: <StringArray>
['A']
Length: 1, dtype: str

Indicator breakdown:
indicator
VALUE_IN_EUROS        82591
QUANTITY_IN_TONNES    82591
Name: count, dtype: int64

Material breakdown:
material
iron_steel     122820
aluminium       25502
fertilizers     12286
cement           4108
hydrogen          466
Name: count, dtype: int64

Years present: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Rows per year:
year
2020    31104
2021    32536
2022    34390
2023    33828
2024    33324
Name: count, dtype: int64


In [24]:
# Trade flows use ISO2 country codes for partner countries.
# Flag any non-standard codes — Eurostat sometimes includes aggregate
# codes (e.g. WORLD, EU27_2020) which would need to be excluded from analysis.
# Drop nulls before sorting — any null partner codes are flagged separately.
null_partners = flows["partner"].isnull().sum()
print(f"Null partner codes: {null_partners}")

partners = sorted(flows["partner"].dropna().unique())
print("Distinct partner countries:", len(partners))

non_standard = [p for p in partners if len(p) != 2]
print("\nNon-standard partner codes (not 2-char ISO2):", non_standard)

print("\nAll partner codes:")
print(partners)

Null partner codes: 178
Distinct partner countries: 243

Non-standard partner codes (not 2-char ISO2): ['EXT_EU27_2020', 'INT_EU27_2020']

All partner codes:
['AD', 'AE', 'AF', 'AG', 'AI', 'AL', 'AM', 'AO', 'AQ', 'AR', 'AS', 'AT', 'AU', 'AW', 'AZ', 'BA', 'BB', 'BD', 'BE', 'BF', 'BG', 'BH', 'BI', 'BJ', 'BL', 'BM', 'BN', 'BO', 'BQ', 'BR', 'BS', 'BT', 'BV', 'BW', 'BY', 'BZ', 'CA', 'CC', 'CD', 'CF', 'CG', 'CH', 'CI', 'CK', 'CL', 'CM', 'CN', 'CO', 'CR', 'CU', 'CV', 'CW', 'CX', 'CY', 'CZ', 'DE', 'DJ', 'DK', 'DM', 'DO', 'DZ', 'EC', 'EE', 'EG', 'EH', 'ER', 'ES', 'ET', 'EXT_EU27_2020', 'FI', 'FJ', 'FK', 'FM', 'FO', 'FR', 'GA', 'GB', 'GD', 'GE', 'GH', 'GI', 'GL', 'GM', 'GN', 'GQ', 'GR', 'GS', 'GT', 'GU', 'GW', 'GY', 'HK', 'HN', 'HR', 'HT', 'HU', 'ID', 'IE', 'IL', 'IN', 'INT_EU27_2020', 'IO', 'IQ', 'IR', 'IS', 'IT', 'JM', 'JO', 'JP', 'KE', 'KG', 'KH', 'KI', 'KM', 'KN', 'KP', 'KR', 'KW', 'KY', 'KZ', 'LA', 'LB', 'LC', 'LI', 'LK', 'LR', 'LS', 'LT', 'LU', 'LV', 'LY', 'MA', 'MD', 'ME', 'MG', 'MH', 'MK

In [25]:
# CN codes are stored as integers in the trade flow data.
# Check digit length distribution — expect a mix of 4, 6 and 8-digit codes,
# consistent with the CBAM defaults which also cover multiple CN levels.
print("Distinct CN codes (product field):", flows["product"].nunique())

flows["cn_len"] = flows["product"].astype(str).str.len()
print("\nCN code length distribution:")
print(flows["cn_len"].value_counts().sort_index())

print("\nExample of each length:")
for length in sorted(flows["cn_len"].unique()):
    sample = flows[flows["cn_len"] == length]["product"].iloc[0]
    print(f"  {length}-digit: {sample}")

Distinct CN codes (product field): 259

CN code length distribution:
cn_len
4    22502
6    42832
8    99848
Name: count, dtype: int64

Example of each length:
  4-digit: 7201
  6-digit: 721123
  8-digit: 26011200


In [26]:
# Check for anomalous values — zeros and negatives are unexpected in import flows.
# Top partners by total EUR value gives a sense of which countries dominate
# EU steel and CBAM-relevant imports over the 2020-2024 period.
print("Value distribution (all indicators):")
print(flows["value"].describe())
print("\nZero-value rows:", (flows["value"] == 0).sum())
print("Negative-value rows:", (flows["value"] < 0).sum())

top = (flows[flows["indicator"] == "VALUE_IN_EUROS"]
       .groupby("partner")["value"]
       .sum()
       .sort_values(ascending=False)
       .head(15))
print("\nTop 15 partners by total import value (EUR, all years):")
print(top.apply(lambda x: f"{x:,.0f}").to_string())

Value distribution (all indicators):
count        165182.0000
mean       19278217.7317
std       247547524.2728
min               0.0000
25%              31.6517
50%            3029.0230
75%          282565.7500
max     20969068554.0000
Name: value, dtype: float64

Zero-value rows: 1747
Negative-value rows: 0

Top 15 partners by total import value (EUR, all years):
partner
INT_EU27_2020    1,095,540,947,589
EXT_EU27_2020      495,253,174,149
DE                 229,175,199,202
IT                 126,654,076,678
NL                 103,563,007,137
BE                  93,349,068,461
FR                  81,776,474,967
CN                  77,611,061,065
ES                  65,796,388,745
PL                  63,888,309,203
AT                  59,586,991,370
TR                  48,694,563,017
CZ                  43,614,785,928
RU                  41,028,403,235
SE                  37,269,502,891


In [41]:
# Trade flows use ISO2 codes; CBAM defaults use full country names;
# Ember has both ISO3 and full names. These formats cannot be joined
# directly — a crosswalk table is required in 07_clean_and_align.ipynb.
# Ember's ISO3 codes provide a bridge: ISO2 <-> ISO3 <-> country name
# via the pycountry library.
print("Trade flow partner format (sample):", sorted(flows["partner"].dropna().unique())[:10])
print("\nCBAM defaults country format (sample):", sorted(defaults["country"].unique())[:10])

print(f"\nCBAM default countries: {defaults['country'].nunique()}")
print(f"Trade flow partner codes: {flows['partner'].nunique()}")
print("\nNote: overlap cannot be confirmed without a crosswalk. Flagged for notebook 07.")

Trade flow partner format (sample): ['AD', 'AE', 'AF', 'AG', 'AI', 'AL', 'AM', 'AO', 'AQ', 'AR']

CBAM defaults country format (sample): ['Albania', 'Algeria', 'Angola', 'Argentina', 'Armenia', 'Australia', 'Azerbaijan', 'Bahrain', 'Bangladesh', 'Belarus']

CBAM default countries: 119
Trade flow partner codes: 243

Note: overlap cannot be confirmed without a crosswalk. Flagged for notebook 07.


### 4. Observations

**Structure**
- 165,182 rows, 9 columns. One row per (partner, product, indicator, year)
  combination. Indicators split evenly: 82,591 VALUE_IN_EUROS rows and
  82,591 QUANTITY_IN_TONNES rows, confirming a clean paired structure.
- reporter is always EU27_2020, flow is always 1 (imports only), freq is
  always A (annual). These columns carry no analytical variance and can be
  dropped in 07.
- Years 2020-2024 all present, row counts broadly stable across years.

**Nulls**
- 178 null partner codes. These need to be investigated and dropped in 07
  as they cannot be joined to any country dimension.

**Non-standard partner codes**
- Two Eurostat aggregate codes present: `EXT_EU27_2020` (extra-EU trade)
  and `INT_EU27_2020` (intra-EU trade). These are not individual countries
  and must be excluded from any country-level analysis. They are also the
  two largest "partners" by value, sitting far above any individual country,
  which confirms they are aggregates rather than data errors.
- Several non-ISO2 codes present in the full list: QP, QV, QW, QY, QZ, XC,
  XL, XS are Eurostat special codes for confidential or unallocated trade.
  Flag for cleaning in 07.

**CN code format**
- Stored as integers. Three lengths present: 4-digit (22,502 rows), 6-digit
  (42,832 rows), 8-digit (99,848 rows). Consistent with CBAM defaults.
  Must be converted to zero-padded strings before joining on CN code.

**Value ranges**
- 1,747 zero-value rows. These represent reported flows with no recorded
  value — likely suppressed for confidentiality or genuinely zero-value
  shipments. Flag for review in 07.
- Max value of ~€20.9bn in a single row is plausible for an aggregate
  flow but worth verifying it is not a duplicate or data entry error.
- Large std dev relative to mean confirms a highly skewed distribution
  typical of trade data — a small number of large flows dominate.

**Country identifier format**
- Trade flows use ISO2 codes; CBAM defaults use full country names; Ember
  has both ISO3 and full names. A crosswalk is required before any join.
  243 partner codes vs 119 CBAM default countries — the trade flows cover
  a much broader set of origins, many of which will have no CBAM default
  value and will need to be handled accordingly in the analysis layer.

## 5. Country Grid Electricity 

(`country_grid_electricity.csv`)

Ember yearly electricity data filtered to countries only. Three variable
types retained: grid CO2 intensity (gCO2/kWh), installed capacity by fuel
type (GW), and generation by fuel type (TWh and % share). Long format —
one row per (country, year, category, variable, unit) combination.

In [29]:
# Basic structure check — this is the largest dataset at ~194k rows
print("Shape:", grid.shape)
print("\nDtypes:")
print(grid.dtypes)
print("\nSample:")
grid.head(3)

Shape: (193936, 10)

Dtypes:
Area                str
ISO 3 code          str
Year              int64
Continent           str
Ember region        str
Category            str
Subcategory         str
Variable            str
Unit                str
Value           float64
dtype: object

Sample:


,Area,ISO 3 code,Year,Continent,Ember region,Category,Subcategory,Variable,Unit,Value
0,Afghanistan,AFG,2000,Asia,Asia,Capacity,Aggregate fuel,Clean,GW,0.1900
1,Afghanistan,AFG,2000,Asia,Asia,Capacity,Aggregate fuel,Fossil,GW,0.0300
2,Afghanistan,AFG,2000,Asia,Asia,Capacity,Aggregate fuel,Renewables,GW,0.1900


In [30]:
# Check for nulls across all columns
print("Nulls:")
print(grid.isnull().sum())

Nulls:
Area                0
ISO 3 code          0
Year                0
Continent           0
Ember region        0
Category            0
Subcategory         0
Variable            0
Unit                0
Value           18656
dtype: int64


In [31]:
# Confirm the three retained variable categories and their units.
# CO2 intensity is the primary variable for CBAM indirect emissions calculations.
print("Category values:", grid["Category"].unique())
print("\nYear range:", grid["Year"].min(), "to", grid["Year"].max())
print("\nVariable breakdown by category:")
print(grid.groupby(["Category", "Variable", "Unit"]).size()
      .reset_index(name="rows").to_string(index=False))

Category values: <StringArray>
['Capacity', 'Electricity generation', 'Power sector emissions']
Length: 3, dtype: str

Year range: 2000 to 2025

Variable breakdown by category:
              Category         Variable     Unit  rows
              Capacity        Bioenergy       GW  5281
              Capacity            Clean       GW  5322
              Capacity             Coal       GW  5213
              Capacity           Fossil       GW  5322
              Capacity              Gas       GW  5142
              Capacity            Hydro       GW  5228
              Capacity          Nuclear       GW  4904
              Capacity     Other Fossil       GW  5321
              Capacity Other Renewables       GW  4676
              Capacity       Renewables       GW  5322
              Capacity            Solar       GW  5304
              Capacity             Wind       GW  5106
Electricity generation        Bioenergy        %  5372
Electricity generation        Bioenergy      TWh  537

In [32]:
# Ember provides both full country names and ISO3 codes.
# ISO3 is the bridge between trade flow ISO2 codes and CBAM default country names.
# Check for any areas without an ISO3 code — these are likely aggregates
# that were not fully filtered out during extraction.
print("Distinct country names (Area):", grid["Area"].nunique())
print("Distinct ISO3 codes:", grid["ISO 3 code"].nunique())

print("\nRows with null ISO3 code:")
print(grid[grid["ISO 3 code"].isnull()][["Area", "Continent", "Ember region"]]
      .drop_duplicates().to_string(index=False))

# Confirm each Area maps to exactly one ISO3 code
iso_check = grid.groupby("Area")["ISO 3 code"].nunique()
print("\nAreas with more than one ISO3 code:")
print(iso_check[iso_check > 1])

Distinct country names (Area): 215
Distinct ISO3 codes: 215

Rows with null ISO3 code:
Empty DataFrame
Columns: [Area, Continent, Ember region]
Index: []

Areas with more than one ISO3 code:
Series([], Name: ISO 3 code, dtype: int64)


In [39]:
# Isolate CO2 intensity rows — the primary variable for CBAM calculations.
# Check value ranges, flag outliers, and confirm how many countries have
# recent data. Stale data may affect indirect emissions estimates.
intensity = grid[
    (grid["Category"] == "Power sector emissions") &
    (grid["Variable"] == "CO2 intensity")
].copy()

print("CO2 intensity rows:", len(intensity))
print("Unit:", intensity["Unit"].unique())
print("\nValue distribution (gCO2/kWh):")
print(intensity["Value"].describe())

print("\nZero or negative intensity rows:")
zero_neg = intensity[intensity["Value"] <= 0]
print(zero_neg[["Area", "Year", "Value"]].to_string(index=False)
      if len(zero_neg) > 0 else "None")

print("\nCountries with intensity > 1000 gCO2/kWh (any year):")
high = intensity[intensity["Value"] > 1000]
print(high[["Area", "Year", "Value"]].sort_values("Value", ascending=False)
      .to_string(index=False) if len(high) > 0 else "None")

CO2 intensity rows: 5407
Unit: <StringArray>
['gCO2/kWh']
Length: 1, dtype: str

Value distribution (gCO2/kWh):
count   5381.0000
mean     474.5117
std      250.2675
min        0.0000
25%      277.5900
50%      525.3900
75%      652.1700
max     1306.7200
Name: Value, dtype: float64

Zero or negative intensity rows:
                          Area  Year  Value
                       Burundi  2000 0.0000
                       Burundi  2001 0.0000
                       Burundi  2002 0.0000
                       Burundi  2003 0.0000
                       Burundi  2004 0.0000
                       Burundi  2005 0.0000
                       Burundi  2006 0.0000
                       Burundi  2007 0.0000
                       Burundi  2008 0.0000
                       Burundi  2009 0.0000
                       Burundi  2012 0.0000
Central African Republic (the)  2013 0.0000
Central African Republic (the)  2014 0.0000
Central African Republic (the)  2015 0.0000
Central African Republ

In [40]:
# For CBAM purposes, the most recent available intensity figure per country
# is what matters. Flag any countries whose latest data predates 2024
latest = intensity.groupby("Area")["Year"].max()
print("Most recent intensity year distribution:")
print(latest.value_counts().sort_index())

stale = latest[latest < 2024].reset_index()
stale.columns = ["Area", "latest_year"]
print(f"\nCountries with latest intensity data before 2024: {len(stale)}")
if len(stale) > 0:
    print(stale.to_string(index=False))

Most recent intensity year distribution:
Year
2009      1
2019      1
2022      1
2023     16
2024    106
2025     90
Name: count, dtype: int64

Countries with latest intensity data before 2024: 19
                                        Area  latest_year
              Central African Republic (the)         2023
                               Comoros (the)         2023
                                    Dominica         2023
           Falkland Islands (the) [Malvinas]         2023
                         Faroe Islands (the)         2023
                               French Guiana         2023
                                  Guadeloupe         2023
                                  Martinique         2023
                                        Niue         2023
                                    Pakistan         2019
                                     Reunion         2023
Saint Helena, Ascension and Tristan da Cunha         2023
                   Saint Pierre and Miquelon    

### 5. Observations

**Structure**
- 193,936 rows, 10 columns. Clean long-format structure with no null
  identifiers. 18,656 null values in the Value column — expected, as not
  every country has data for every fuel type and year combination.
- Three categories retained: Capacity (GW), Electricity generation (TWh
  and % share), and Power sector emissions (CO2 intensity, gCO2/kWh).
- Year range 2000-2025, though 2025 coverage is partial.

**Country identifiers**
- 215 distinct countries, each with exactly one ISO3 code. No nulls, no
  duplicates. Ember's ISO3 column is the crosswalk anchor for joining
  trade flow ISO2 codes to CBAM default country names in 07.

**CO2 intensity — value ranges**
- 5,407 intensity rows, 26 nulls within the subset (early years with no
  reported data for some countries).
- Mean 474 gCO2/kWh, range 0 to 1,307 gCO2/kWh. Wide spread expected
  given the mix of coal-heavy and renewable-heavy grids globally.
- Zero-value rows: Burundi (2000-2012, non-consecutive) and Central
  African Republic (2013-2023) both report 0.0 gCO2/kWh. Both countries
  rely almost entirely on hydro, which makes near-zero plausible, but
  exact zeros across many years suggest a reporting floor rather than
  genuine measurement. Treat as near-zero rather than null in 07.
- High outliers: Turkmenistan is the dataset maximum at ~1,307 gCO2/kWh
  consistently across all years — an almost entirely gas-flaring grid with
  no meaningful renewables. Uzbekistan follows at 1,040-1,150 gCO2/kWh.
  Both are physically plausible and consistent with known grid composition.
  No action needed.

**Data recency**
- 196 of 215 countries have intensity data up to 2024 or 2025. Well-covered
  for CBAM purposes.
- 19 countries have latest data before 2024. Most are at 2023 (16 countries),
  which is acceptable. Two cases warrant flagging:
  - Pakistan (latest year 2019): a major CBAM-relevant exporter with stale
    intensity data. The 2019 figure will need to be used as-is or flagged
    as an approximation in the analysis.
  - Western Sahara (latest year 2009): not a significant CBAM trade partner,
    low risk.
  - Ukraine (latest year 2022): understandable given the ongoing conflict
    affecting data collection. Use 2022 figure with a note.
- Country names in Ember use formal UN-style names in some cases
  (e.g. "Central African Republic (the)", "Comoros (the)") which will not
  match CBAM defaults or pycountry lookups directly. Flag for standardization
  in 07.

## 6. Cross-Dataset Summary

Assessment of join readiness across all datasets. Identifies the key
mismatches that must be resolved in 07_clean_and_align.ipynb before
the database can be built.

In [42]:
# Each dataset uses a different country identifier format.
# Summarize the format used by each and flag what crosswalk is needed.
summary = [
    {"dataset": "cbam_defaults",            "country_format": "Full name (e.g. 'Albania')",         "notes": "119 countries"},
    {"dataset": "eu_import_trade_flows",    "country_format": "ISO2 (e.g. 'AL')",                   "notes": "243 partners, incl. aggregates and nulls"},
    {"dataset": "country_grid_electricity", "country_format": "Full name + ISO3 (e.g. 'ALB')",      "notes": "215 countries, some UN-style formal names"},
    {"dataset": "hydrogen_route_intensities","country_format": "No country dimension",               "notes": "Global averages only"},
    {"dataset": "steel_route_intensity",    "country_format": "No country dimension",               "notes": "Global averages only"},
]
print(pd.DataFrame(summary).to_string(index=False))
print("\nCrosswalk strategy:")
print("  ISO2 (flows) -> ISO3 (Ember) -> country name (defaults) via pycountry.")
print("  Ember ISO3 is the bridge. Build crosswalk in 07 before any join.")

                   dataset                country_format                                     notes
             cbam_defaults    Full name (e.g. 'Albania')                             119 countries
     eu_import_trade_flows              ISO2 (e.g. 'AL')  243 partners, incl. aggregates and nulls
  country_grid_electricity Full name + ISO3 (e.g. 'ALB') 215 countries, some UN-style formal names
hydrogen_route_intensities          No country dimension                      Global averages only
     steel_route_intensity          No country dimension                      Global averages only

Crosswalk strategy:
  ISO2 (flows) -> ISO3 (Ember) -> country name (defaults) via pycountry.
  Ember ISO3 is the bridge. Build crosswalk in 07 before any join.


In [47]:
# CN codes are stored differently across datasets.
# All must be standardized to a consistent format before joining.
cn_summary = [
    {"dataset": "cbam_defaults",         "cn_format": "Spaced string",  "example": "'2523 10 00'", "lengths": "4, 6, 8 digit"},
    {"dataset": "eu_import_trade_flows", "cn_format": "Integer",        "example": "26011200",     "lengths": "4, 6, 8 digit"},
    {"dataset": "hydrogen_route_intensities", "cn_format": "Spaced string", "example": "'2804 10 00'", "lengths": "8 digit only"},
]
print(pd.DataFrame(cn_summary).to_string(index=False))

                   dataset     cn_format      example       lengths
             cbam_defaults Spaced string '2523 10 00' 4, 6, 8 digit
     eu_import_trade_flows       Integer     26011200 4, 6, 8 digit
hydrogen_route_intensities Spaced string '2804 10 00'  8 digit only


In [44]:
# Summary of all datasets, their row counts, and the join keys
# available for linking them in the database layer.
join_summary = pd.DataFrame([
    {"dataset": "cbam_defaults",             "rows": len(defaults),  "join_keys": "country (name), cn_code"},
    {"dataset": "eu_import_trade_flows",     "rows": len(flows),     "join_keys": "partner (ISO2), product (int CN code), year"},
    {"dataset": "country_grid_electricity",  "rows": len(grid),      "join_keys": "Area (name), ISO 3 code, Year"},
    {"dataset": "hydrogen_route_intensities","rows": len(hydrogen),  "join_keys": "cn_code, feedstock_type"},
    {"dataset": "steel_route_intensity",     "rows": len(steel),     "join_keys": "production_route, year"},
])
print(join_summary.to_string(index=False))

                   dataset   rows                                   join_keys
             cbam_defaults  10671                     country (name), cn_code
     eu_import_trade_flows 165182 partner (ISO2), product (int CN code), year
  country_grid_electricity 193936               Area (name), ISO 3 code, Year
hydrogen_route_intensities      6                     cn_code, feedstock_type
     steel_route_intensity     12                      production_route, year


## 7. Issue Log 

To be implemented in: `07_clean_and_align.ipynb`

Findings from this assessment, in priority order.

**Critical — join blockers:**

1. Country identifier formats differ across all three country-level datasets.
   A crosswalk (ISO2 / ISO3 / country name) must be built before any join.
   Anchor on Ember ISO3 codes via pycountry.
2. CN code formats differ between datasets (spaced strings vs integers).
   Standardize to unspaced strings.
3. `eu_import_trade_flows` contains 178 null partner codes, two Eurostat
   aggregate codes (INT/EXT_EU27_2020), and several non-ISO2 special codes
   (QP, QV, QW, QY, QZ, XC, XL, XS).
   Map to labels in 07 and decide on inclusion per code.

**Data quality — fix in 07:**

4. `cbam_defaults.production_route`: replace letter codes with full names
   per the benchmark annex key. Retain original code column for joining to
   steel_route_intensity. Also fix `'\xa0'` non-breaking space values.
5. `cbam_defaults`: 1 row with zero direct_emissions and total_emissions.
   Investigate and handle.
6. `cbam_defaults`: `default_2026` has 1 null. Investigate source row.
7. `cbam_defaults`: country name issues to standardize — "Democratic
   Republic of the Cong" (truncated), "Myanmar_Burma" (underscore),
   "Côte d'Ivoire" and "Curaçao" (confirm encoding).
8. `country_grid_electricity`: Ember uses UN-style formal names for some
   countries (e.g. "Central African Republic (the)"). These will not match
   pycountry lookups directly and need manual mapping.
9. `country_grid_electricity`: Burundi and Central African Republic have
   zero CO2 intensity values across multiple years. Treat as near-zero
   rather than null — do not drop.
10. `country_grid_electricity`: Pakistan (2019), Ukraine (2022), and
    Western Sahara (2009) have stale intensity data. Use latest available
    with a note in the analysis layer.
11. `eu_import_trade_flows`: convert "A", "1", etc. to human-readable
    field values.

**Expected nulls — no action needed:**

12. `cbam_defaults.indirect_emissions`: 74.3% null. Expected — electricity
    and hydrogen sectors carry no indirect component.
13. `cbam_defaults.production_route`: 9.7% null. Expected — non-steel
    sectors do not use route codes.
14. `hydrogen_route_intensities.comments`: 3 nulls. Sparse by design.

**Deferred to analysis layer:**

15. Which year of steel_route_intensity to use as the reference intensity
    for joining to cbam_defaults (2024, or average across 2022-2024).
16. Once the country crosswalk is built, verify overlap between 
    CBAM default countries and trade flow partners. Document any countries 
    present in defaults but absent from trade flows, and vice versa.